In [1]:
from pathlib import Path
import re

# =========================================================
# CHANGE ONLY THESE PATHS IF NEEDED
# =========================================================

PURE_SOURCE = Path(
    "/media/data/rPPG/rPPG_Data/PURE/ALL/ALL"
)

UBFC_SOURCE = Path(
    "/media/data/rPPG/rPPG_Data/UBFC_rPPG"
)

OUTPUT_VIEW = Path(
    "/media/data/rPPG/rPPG_Data/RhythmMamba_DataView"
)


def create_safe_link(source: Path, destination: Path, is_directory=False):
    """Create a symbolic link without changing the original data."""

    source = source.resolve()

    if not source.exists():
        raise FileNotFoundError(f"Source does not exist: {source}")

    # If the correct link already exists, do nothing.
    if destination.is_symlink():
        if destination.resolve() == source:
            return
        raise RuntimeError(
            f"Different symbolic link already exists: {destination}"
        )

    # Do not overwrite a real file or directory.
    if destination.exists():
        raise FileExistsError(
            f"Destination already exists and will not be overwritten: "
            f"{destination}"
        )

    try:
        destination.symlink_to(
            source,
            target_is_directory=is_directory
        )
    except PermissionError as error:
        raise PermissionError(
            "Symbolic-link permission was denied. "
            "On Windows, enable Developer Mode or run Python as "
            "Administrator."
        ) from error


# =========================================================
# CHECK SOURCE PATHS
# =========================================================

if not PURE_SOURCE.is_dir():
    raise FileNotFoundError(f"PURE path not found: {PURE_SOURCE}")

if not UBFC_SOURCE.is_dir():
    raise FileNotFoundError(f"UBFC path not found: {UBFC_SOURCE}")


# =========================================================
# CREATE PURE COMPATIBILITY VIEW
# Expected:
# PURE/01-01/01-01/*.png
# PURE/01-01/01-01.json
# =========================================================

pure_output = OUTPUT_VIEW / "PURE"
pure_output.mkdir(parents=True, exist_ok=True)

pure_recordings = sorted(
    path for path in PURE_SOURCE.iterdir()
    if path.is_dir()
    and re.fullmatch(r"\d{2}-\d{2}", path.name)
)

pure_created = 0

for recording_directory in pure_recordings:
    recording_name = recording_directory.name
    json_source = PURE_SOURCE / f"{recording_name}.json"

    if not json_source.is_file():
        print(f"WARNING: Missing PURE JSON: {json_source}")
        continue

    recording_output = pure_output / recording_name
    recording_output.mkdir(parents=True, exist_ok=True)

    create_safe_link(
        recording_directory,
        recording_output / recording_name,
        is_directory=True
    )

    create_safe_link(
        json_source,
        recording_output / f"{recording_name}.json",
        is_directory=False
    )

    pure_created += 1


# =========================================================
# CREATE UBFC-rPPG COMPATIBILITY VIEW
# Expected:
# UBFC/subject1/vid.avi
# UBFC/subject1/ground_truth.txt
# =========================================================

ubfc_output = OUTPUT_VIEW / "UBFC"
ubfc_output.mkdir(parents=True, exist_ok=True)

ubfc_recordings = []

for path in UBFC_SOURCE.iterdir():
    match = re.fullmatch(r"vid_(\d+)", path.name)

    if path.is_dir() and match:
        subject_number = int(match.group(1))
        ubfc_recordings.append((subject_number, path))

ubfc_recordings.sort(key=lambda item: item[0])

ubfc_created = 0

for subject_number, recording_directory in ubfc_recordings:
    video_source = (
        recording_directory / f"vid_{subject_number}.avi"
    )
    ground_truth_source = (
        recording_directory / f"ground_truth_{subject_number}.txt"
    )

    if not video_source.is_file():
        print(f"WARNING: Missing UBFC video: {video_source}")
        continue

    if not ground_truth_source.is_file():
        print(
            f"WARNING: Missing UBFC ground truth: "
            f"{ground_truth_source}"
        )
        continue

    subject_output = ubfc_output / f"subject{subject_number}"
    subject_output.mkdir(parents=True, exist_ok=True)

    create_safe_link(
        video_source,
        subject_output / "vid.avi",
        is_directory=False
    )

    create_safe_link(
        ground_truth_source,
        subject_output / "ground_truth.txt",
        is_directory=False
    )

    ubfc_created += 1


# =========================================================
# FINAL VERIFICATION
# =========================================================

print("=" * 70)
print("RhythmMamba compatibility view created successfully")
print("=" * 70)

print(f"PURE recordings found   : {len(pure_recordings)}")
print(f"PURE recordings prepared: {pure_created}")

print(f"UBFC recordings found   : {len(ubfc_recordings)}")
print(f"UBFC recordings prepared: {ubfc_created}")

print("\nUse these paths in the RhythmMamba configuration:")

print(f'PURE DATA_PATH: "{pure_output}"')
print(f'UBFC DATA_PATH: "{ubfc_output}"')

print("\nExample PURE layout:")
for item in sorted((pure_output / "01-01").iterdir()):
    print(" ", item)

print("\nExample UBFC layout:")
for item in sorted((ubfc_output / "subject1").iterdir()):
    print(" ", item)

RhythmMamba compatibility view created successfully
PURE recordings found   : 59
PURE recordings prepared: 59
UBFC recordings found   : 42
UBFC recordings prepared: 42

Use these paths in the RhythmMamba configuration:
PURE DATA_PATH: "/media/data/rPPG/rPPG_Data/RhythmMamba_DataView/PURE"
UBFC DATA_PATH: "/media/data/rPPG/rPPG_Data/RhythmMamba_DataView/UBFC"

Example PURE layout:
  /media/data/rPPG/rPPG_Data/RhythmMamba_DataView/PURE/01-01/01-01
  /media/data/rPPG/rPPG_Data/RhythmMamba_DataView/PURE/01-01/01-01.json

Example UBFC layout:
  /media/data/rPPG/rPPG_Data/RhythmMamba_DataView/UBFC/subject1/ground_truth.txt
  /media/data/rPPG/rPPG_Data/RhythmMamba_DataView/UBFC/subject1/vid.avi


In [1]:
import platform
import shutil
import subprocess
import sys
import time

print("=" * 70)
print("SYSTEM INFORMATION")
print("=" * 70)
print("Operating system :", platform.platform())
print("Python version   :", sys.version.split()[0])

# ============================================================
# NVIDIA DRIVER AND POWER INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("NVIDIA DRIVER / POWER INFORMATION")
print("=" * 70)

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi is None:
    # Common Linux location
    possible_path = "/usr/bin/nvidia-smi"
    if shutil.which(possible_path):
        nvidia_smi = possible_path

if nvidia_smi is not None:
    command = [
        nvidia_smi,
        "--query-gpu=index,name,driver_version,memory.total,"
        "power.draw,power.limit",
        "--format=csv,noheader",
    ]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True,
        )

        print(
            "GPU index, GPU name, driver, memory, "
            "current power, maximum power"
        )
        print(result.stdout.strip())

    except Exception as error:
        print("nvidia-smi was found but the detailed query failed.")
        print("Error:", error)
else:
    print("nvidia-smi was not found in the system PATH.")


# ============================================================
# PYTORCH GPU CHECK
# ============================================================

print("\n" + "=" * 70)
print("PYTORCH AND CUDA CHECK")
print("=" * 70)

try:
    import torch

    print("PyTorch version       :", torch.__version__)
    print("PyTorch CUDA version  :", torch.version.cuda)
    print("CUDA available        :", torch.cuda.is_available())
    print("cuDNN available       :", torch.backends.cudnn.is_available())
    print("cuDNN version         :", torch.backends.cudnn.version())
    print("Number of GPUs        :", torch.cuda.device_count())

    if not torch.cuda.is_available():
        print(
            "\nPyTorch cannot currently access the GPU. "
            "This may mean that the base environment has a CPU-only "
            "PyTorch installation."
        )

    else:
        for gpu_index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(gpu_index)
            free_memory, total_memory = torch.cuda.mem_get_info(gpu_index)

            print("\n" + "-" * 70)
            print(f"GPU {gpu_index}")
            print("-" * 70)

            print("Name               :", properties.name)
            print(
                "Total memory       :",
                f"{properties.total_memory / 1024**3:.2f} GB",
            )
            print(
                "Currently free     :",
                f"{free_memory / 1024**3:.2f} GB",
            )
            print(
                "Compute capability :",
                f"{properties.major}.{properties.minor}",
            )
            print(
                "Multiprocessors     :",
                properties.multi_processor_count,
            )

            # ----------------------------------------------------
            # Small GPU calculation test
            # ----------------------------------------------------

            device = torch.device(f"cuda:{gpu_index}")

            matrix_size = (
                4096 if properties.total_memory >= 4 * 1024**3
                else 2048
            )
            repetitions = 5

            print(
                f"\nRunning {matrix_size} × {matrix_size} "
                "matrix-multiplication test..."
            )

            try:
                a = torch.randn(
                    matrix_size,
                    matrix_size,
                    device=device,
                    dtype=torch.float32,
                )
                b = torch.randn(
                    matrix_size,
                    matrix_size,
                    device=device,
                    dtype=torch.float32,
                )

                # Warm-up
                for _ in range(2):
                    _ = a @ b

                torch.cuda.synchronize(gpu_index)

                start_time = time.perf_counter()

                for _ in range(repetitions):
                    result = a @ b

                torch.cuda.synchronize(gpu_index)
                elapsed_time = time.perf_counter() - start_time

                average_time = elapsed_time / repetitions

                approximate_tflops = (
                    2 * matrix_size**3 / average_time / 1e12
                )

                print("GPU calculation     : PASSED")
                print(
                    "Average time         :",
                    f"{average_time:.4f} seconds",
                )
                print(
                    "Approx. FP32 speed   :",
                    f"{approximate_tflops:.2f} TFLOPS",
                )

                del a, b, result
                torch.cuda.empty_cache()

            except Exception as benchmark_error:
                print("GPU calculation     : FAILED")
                print("Error:", benchmark_error)

except ImportError:
    print("PyTorch is not installed in the current Python environment.")

SYSTEM INFORMATION
Operating system : Linux-6.8.0-111-generic-x86_64-with-glibc2.35
Python version   : 3.11.15

NVIDIA DRIVER / POWER INFORMATION
nvidia-smi was found but the detailed query failed.
Error: Command '['/usr/bin/nvidia-smi', '--query-gpu=index,name,driver_version,memory.total,power.draw,power.limit', '--format=csv,noheader']' returned non-zero exit status 18.

PYTORCH AND CUDA CHECK
PyTorch version       : 2.1.2+cu121
PyTorch CUDA version  : 12.1
CUDA available        : True
cuDNN available       : True
cuDNN version         : 8902
Number of GPUs        : 2

----------------------------------------------------------------------
GPU 0
----------------------------------------------------------------------
Name               : Tesla V100-PCIE-32GB
Total memory       : 31.74 GB
Currently free     : 31.00 GB
Compute capability : 7.0
Multiprocessors     : 80

Running 4096 × 4096 matrix-multiplication test...


/home/rafsan/miniconda3/envs/mamba_hunting/lib/python3.11/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


GPU calculation     : PASSED
Average time         : 0.0141 seconds
Approx. FP32 speed   : 9.78 TFLOPS

----------------------------------------------------------------------
GPU 1
----------------------------------------------------------------------
Name               : Tesla V100-PCIE-32GB
Total memory       : 31.74 GB
Currently free     : 31.00 GB
Compute capability : 7.0
Multiprocessors     : 80

Running 4096 × 4096 matrix-multiplication test...
GPU calculation     : PASSED
Average time         : 0.0139 seconds
Approx. FP32 speed   : 9.91 TFLOPS
